In [1]:
#!/usr/bin/env python3
"""
Corner plot + interactive 3D scatter from an Excel file.

Required columns (exact names):
- "Delta Lc,nm"
- "Contour length,nm"
- "Force, pN"

Outputs:
- <output_dir>/corner_plot.png and .svg
- <output_dir>/interactive_3d_scatter.html (rotatable, zoomable)
- (optional) static PNG/SVG of the 3D view if kaleido is installed
"""

import os
import argparse
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Plotly for interactive 3D
import plotly.express as px



In [2]:

# ---------------------------
# Config defaults (tweakable)
# ---------------------------
FONT_SIZE = 14
CORNER_FIGSIZE = (9, 9)
POINT_ALPHA = 0.7
POINT_SIZE = 20  # for seaborn (in points^2)
PLOT_TITLE = "Corner & 3D visualization"

REQUIRED_COLS = ["Delta Lc,nm", "Contour length,nm", "Force, pN"]


In [3]:

# ---------------------------
# Data utilities
# ---------------------------
def load_excel_required_columns(file_path, required_cols):
    """Load Excel and return a clean DataFrame with the required columns only."""
    df = pd.read_excel(file_path)
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}. Found columns: {list(df.columns)}")
    # Keep just the required columns and drop rows where any is NaN or non-finite
    sub = df[required_cols].apply(pd.to_numeric, errors="coerce")
    sub = sub.replace([np.inf, -np.inf], np.nan).dropna(how="any")
    return sub

# ---------------------------
# Styling helper
# ---------------------------
def format_ax(ax, title=None):
    """Apply consistent formatting to a Matplotlib axis."""
    if title:
        ax.set_title(title, fontsize=FONT_SIZE)
    for side in ["top", "right", "left", "bottom"]:
        ax.spines[side].set_linewidth(1.8)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE, direction="in", length=6, width=1.8)

# ---------------------------
# Corner (pair) plot
# ---------------------------
def corner_plot(df, output_dir, title="Corner plot"):
    """
    Make a corner/pair plot with scatter off-diagonal and hist on the diagonal.
    Saves PNG and SVG.
    """
    os.makedirs(output_dir, exist_ok=True)

    # Seaborn pairplot
    g = sns.pairplot(
        df,
        corner=True,
        plot_kws={"alpha": POINT_ALPHA, "s": POINT_SIZE, "linewidth": 0},
        diag_kind="hist",
        diag_kws={"bins": 40, "alpha": 0.9}
    )
    # Global title
    g.fig.suptitle(title, y=1.02, fontsize=FONT_SIZE + 2)

    # Tight layout and save
    g.fig.set_size_inches(*CORNER_FIGSIZE)
    png_path = os.path.join(output_dir, "corner_plot.png")
    svg_path = os.path.join(output_dir, "corner_plot.svg")
    g.fig.savefig(png_path, dpi=300, bbox_inches="tight")
    g.fig.savefig(svg_path, format="svg", bbox_inches="tight")
    plt.close(g.fig)
    print(f"Corner plot saved:\n- {png_path}\n- {svg_path}")

# ---------------------------
# Interactive 3D plot (Plotly)
# ---------------------------
def interactive_3d_plot(df, output_dir, title="Interactive 3D: ΔLc vs Contour length vs Force",
                        point_size=3):
    """
    Build an interactive 3D scatter with Plotly and save to HTML.
    Optionally also saves static PNG/SVG if kaleido is installed.
    """
    os.makedirs(output_dir, exist_ok=True)

    x, y, z = REQUIRED_COLS  # ["Delta Lc,nm", "Contour length,nm", "Force, pN"]

    fig = px.scatter_3d(
        df,
        x=x, y=y, z=z,
        opacity=0.85,
        title=title,
        height=700
    )
    fig.update_traces(marker=dict(size=point_size))
    fig.update_layout(
        scene=dict(
            xaxis_title=x,
            yaxis_title=y,
            zaxis_title=z
        ),
        title_x=0.5
    )

    html_path = os.path.join(output_dir, "interactive_3d_scatter.html")
    fig.write_html(html_path, include_plotlyjs="cdn", full_html=True)
    print(f"Interactive 3D HTML saved:\n- {html_path}")

    # Optional static export (requires: pip install -U "kaleido>=0.2.1")
    try:
        png_path = os.path.join(output_dir, "interactive_3d_scatter.png")
        svg_path = os.path.join(output_dir, "interactive_3d_scatter.svg")
        fig.write_image(png_path, scale=2)  # needs kaleido
        fig.write_image(svg_path)           # needs kaleido
        print(f"Also saved static 3D snapshots:\n- {png_path}\n- {svg_path}")
    except Exception as e:
        # It's fine if kaleido isn't installed
        print(f"(Static image export skipped: {e})")

# ---------------------------
# Convenience wrapper
# ---------------------------
def run(file_path, output_dir, title=PLOT_TITLE, point_size_3d=3):
    df = load_excel_required_columns(file_path, REQUIRED_COLS)

    # Corner plot
    corner_plot(df, output_dir, title=f"{title} — Corner plot")

    # Interactive 3D
    interactive_3d_plot(
        df,
        output_dir,
        title=f"{title} — 3D scatter",
        point_size=point_size_3d
    )


In [9]:

parser = argparse.ArgumentParser(description="Corner & 3D plots from Excel.")
parser.add_argument("excel", help="Path to the Excel file (.xlsx, .xls)")
parser.add_argument("-o", "--output", default="plots_out", help="Output directory (default: plots_out)")
parser.add_argument("--title", default=PLOT_TITLE, help="Base plot title")
parser.add_argument("--point-size-3d", type=int, default=3, help="Marker size for the 3D plotly scatter")
args = parser.parse_args()


usage: ipykernel_launcher.py [-h] [-o OUTPUT] [--title TITLE]
                             [--point-size-3d POINT_SIZE_3D]
                             excel
ipykernel_launcher.py: error: the following arguments are required: excel


SystemExit: 2

In [ ]:

run(args.excel, args.output, title=args.title, point_size_3d=args.point_size_3d)

In [ ]:
# ---------------------------
# CLI
# ---------------------------
if __name__ == "__main__":
